# Module 10 — Capstone: End-to-End Production RAG System

This notebook assembles **every concept from the curriculum** into one cohesive, production-grade RAG pipeline.

## Architecture
```
┌─────────────────────────────────────────────────────────────────────┐
│                    PRODUCTION RAG PIPELINE                          │
├─────────────────┬───────────────────────────┬───────────────────────┤
│   INGESTION     │      RETRIEVAL            │   GENERATION          │
│                 │                           │                       │
│ PyPDF / Web     │  BM25 (sparse)            │  LangGraph Agent      │
│ CSV / JSON      │  Chroma (dense)     ─────▶│  Query Classification │
│ Text Splitter   │  EnsembleRetriever        │  Conditional Retrieve │
│ Metadata Enrich │  CrossEncoder Rerank      │  Grounding Check      │
│ Chroma Index    │  ContextualCompression    │  Citation Generation  │
└─────────────────┴───────────────────────────┴───────────────────────┘
                                      │
                          ┌───────────▼──────────┐
                          │   RAGAS EVALUATION   │
                          │ Faithfulness         │
                          │ Answer Relevancy     │
                          │ Context Precision    │
                          │ Context Recall       │
                          └──────────────────────┘
```

## Modules Used
| Step | Concept | Module |
|---|---|---|
| Document loading | PyPDF, WebBaseLoader | 2.1 |
| Chunking | RecursiveCharacterTextSplitter | 2.2 |
| Metadata | Enrichment & filtering | 2.4 |
| Embeddings | text-embedding-3-small | 3.2 |
| Vector store | Chroma CRUD | 4.2 |
| Hybrid retrieval | BM25 + Dense Ensemble | 5.4 |
| Re-ranking | CrossEncoder | 6.5 |
| Compression | EmbeddingsFilter | 6.1 |
| Multi-query | Query expansion | 6.4 |
| Agentic RAG | LangGraph state machine | 8.3 |
| Evaluation | RAGAS metrics | 9 |

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import os, hashlib
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

# Verify key is set
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in your .env file'
print('✅ Environment loaded')

## Step 1 — Ingestion Pipeline

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# ── Sample corpus (replace with your own documents) ───────────────────────────
RAW_CORPUS = [
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) is a technique that enhances large language "
            "models by retrieving relevant external documents at inference time. RAG was introduced "
            "to address two fundamental limitations: static knowledge cutoffs and hallucination. "
            "The pipeline has three stages: document ingestion, retrieval, and generation."
        ),
        metadata={"source": "rag_overview.txt", "author": "Course Team", "year": 2024}
    ),
    Document(
        page_content=(
            "LangChain is a framework for building applications powered by language models. "
            "It provides abstractions for document loaders, text splitters, embeddings, vector stores, "
            "retrievers, chains, and agents. LangGraph extends LangChain with graph-based state machines "
            "that enable complex agentic workflows with cycles and conditional branching."
        ),
        metadata={"source": "langchain_overview.txt", "author": "Course Team", "year": 2024}
    ),
    Document(
        page_content=(
            "Vector stores index high-dimensional embeddings and support fast approximate nearest "
            "neighbour search. Popular choices include Chroma (open-source, embedded), FAISS (Meta, "
            "high-performance), Qdrant (production-grade, filtering), and Pinecone (fully managed). "
            "CRUD operations on vector stores allow adding, updating, and deleting chunks."
        ),
        metadata={"source": "vector_stores.txt", "author": "Course Team", "year": 2024}
    ),
    Document(
        page_content=(
            "Hybrid search combines dense (semantic) and sparse (keyword) retrieval. "
            "BM25 matches exact keywords while dense vectors capture semantic similarity. "
            "The EnsembleRetriever in LangChain merges results using Reciprocal Rank Fusion (RRF), "
            "with a configurable weight parameter. Hybrid search outperforms pure dense retrieval "
            "on technical queries, product codes, and named entities."
        ),
        metadata={"source": "hybrid_search.txt", "author": "Course Team", "year": 2024}
    ),
    Document(
        page_content=(
            "RAGAS (Retrieval-Augmented Generation Assessment) is a framework for evaluating RAG "
            "pipelines without requiring human annotations. Key metrics: Faithfulness measures "
            "if the answer is grounded in the context. Answer Relevancy checks if the answer "
            "addresses the question. Context Precision measures if retrieved chunks are relevant. "
            "Context Recall measures if all needed information was retrieved."
        ),
        metadata={"source": "ragas_guide.txt", "author": "Course Team", "year": 2024}
    ),
    Document(
        page_content=(
            "Advanced RAG patterns include: (1) RAG Fusion — generate multiple query variants "
            "and merge results via RRF. (2) HyDE — generate a hypothetical answer and use its "
            "embedding for retrieval. (3) Corrective RAG — grade retrieved docs and fall back to "
            "web search if irrelevant. (4) Self-RAG — use reflection tokens to decide whether to "
            "retrieve, and to verify groundedness of generated answers."
        ),
        metadata={"source": "advanced_rag.txt", "author": "Course Team", "year": 2024}
    ),
]

# ── Split into chunks ─────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50, add_start_index=True
)
chunks = splitter.split_documents(RAW_CORPUS)

# ── Enrich metadata ───────────────────────────────────────────────────────────
enriched = []
for i, c in enumerate(chunks):
    meta = {
        **c.metadata,
        "chunk_id"   : hashlib.md5(c.page_content.encode()).hexdigest()[:8],
        "chunk_index": i,
        "ingested_at": datetime.utcnow().isoformat(),
    }
    enriched.append(Document(page_content=c.page_content, metadata=meta))

print(f'📄 Raw docs     : {len(RAW_CORPUS)}')
print(f'✂  Chunks       : {len(enriched)}')
for c in enriched[:3]:
    print(f'   [{c.metadata["chunk_id"]}] {c.page_content[:70].strip()}...')

In [ ]:
# ── Index into Chroma ─────────────────────────────────────────────────────────
embeddings  = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(
    enriched, embeddings,
    collection_name='capstone',
    persist_directory='./capstone_chroma'
)
print(f'✅ Indexed {len(enriched)} chunks into Chroma')
print(f'   Collection: capstone | Persist dir: ./capstone_chroma')

## Step 2 — Hybrid Retriever with Re-ranking

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from sentence_transformers import CrossEncoder

# ── BM25 (sparse) ─────────────────────────────────────────────────────────────
bm25 = BM25Retriever.from_documents(enriched, k=5)

# ── Dense (semantic) ─────────────────────────────────────────────────────────
dense = vectorstore.as_retriever(search_kwargs={'k': 5})

# ── Ensemble ──────────────────────────────────────────────────────────────────
ensemble = EnsembleRetriever(
    retrievers=[bm25, dense],
    weights=[0.35, 0.65]   # 35% BM25, 65% semantic
)

# ── Compression (filter below 0.75 similarity) ────────────────────────────────
compressor       = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.75)
hybrid_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble
)

# ── Cross-Encoder Re-ranker ───────────────────────────────────────────────────
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_and_rerank(query: str, k: int = 5, top_n: int = 3) -> list:
    # Stage 1: hybrid + compression
    candidates = hybrid_retriever.invoke(query)
    if not candidates:
        return []
    # Stage 2: cross-encoder re-rank
    pairs  = [(query, d.page_content) for d in candidates]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, candidates), reverse=True)
    return [doc for _, doc in ranked[:top_n]]

# ── Test retrieval ────────────────────────────────────────────────────────────
test_query = 'How does hybrid search improve retrieval recall?'
results    = retrieve_and_rerank(test_query)

print(f'Query: "{test_query}"')
print(f'\nTop-{len(results)} chunks after re-ranking:')
for i, doc in enumerate(results, 1):
    print(f'  [{i}] (source: {doc.metadata["source"]}) {doc.page_content[:100].strip()}...')

## Step 3 — Agentic RAG with LangGraph

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# ── State schema ──────────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question        : str
    complexity      : str
    documents       : List[Document]
    relevance_grade : str
    rewritten_query : str
    iterations      : int
    answer          : str
    citations       : List[str]

# ── Nodes ─────────────────────────────────────────────────────────────────────
def classify_node(state):
    prompt = ChatPromptTemplate.from_template(
        "Is this question 'simple' (common knowledge) or 'complex' (needs documents)?\n"
        "Answer only 'simple' or 'complex'.\nQuestion: {question}"
    )
    r = (prompt | llm | StrOutputParser()).invoke({'question': state['question']})
    return {**state, 'complexity': 'complex' if 'complex' in r.lower() else 'simple'}

def retrieve_node(state):
    q    = state.get('rewritten_query') or state['question']
    docs = retrieve_and_rerank(q, top_n=3)
    return {**state, 'documents': docs, 'iterations': state.get('iterations', 0) + 1}

def grade_node(state):
    prompt = ChatPromptTemplate.from_template(
        "Is this document useful for answering '{question}'? Answer yes or no.\nDoc: {doc}"
    )
    relevant = [
        d for d in state['documents']
        if 'yes' in (prompt | llm | StrOutputParser()).invoke(
            {'question': state['question'], 'doc': d.page_content[:300]}
        ).lower()
    ]
    grade = 'relevant' if relevant else 'irrelevant'
    return {**state, 'documents': relevant or state['documents'], 'relevance_grade': grade}

def rewrite_node(state):
    prompt = ChatPromptTemplate.from_template(
        'Rewrite this query to retrieve better documents:\n{question}'
    )
    rewritten = (prompt | llm | StrOutputParser()).invoke({'question': state['question']})
    return {**state, 'rewritten_query': rewritten.strip()}

def direct_answer_node(state):
    prompt = ChatPromptTemplate.from_template('Answer concisely: {question}')
    answer = (prompt | llm | StrOutputParser()).invoke({'question': state['question']})
    return {**state, 'answer': answer, 'citations': []}

def generate_node(state):
    parts     = []
    citations = []
    for i, d in enumerate(state['documents'], 1):
        parts.append(f'[{i}] {d.page_content}')
        citations.append(d.metadata.get('source', f'doc_{i}'))
    context = '\n\n'.join(parts)
    prompt  = ChatPromptTemplate.from_template(
        'Answer ONLY using the numbered context. Cite sources as [N].\n\n'
        'Context:\n{context}\n\nQuestion: {question}'
    )
    answer = (prompt | llm | StrOutputParser()).invoke(
        {'context': context, 'question': state['question']}
    )
    return {**state, 'answer': answer, 'citations': citations}

# ── Routing ───────────────────────────────────────────────────────────────────
def route_complexity(state):
    return 'direct' if state['complexity'] == 'simple' else 'retrieve'

def route_grade(state):
    if state['relevance_grade'] == 'relevant' or state.get('iterations', 0) >= 2:
        return 'generate'
    return 'rewrite'

# ── Build graph ───────────────────────────────────────────────────────────────
build  = StateGraph(RAGState)
build.add_node('classify', classify_node)
build.add_node('retrieve', retrieve_node)
build.add_node('grade',    grade_node)
build.add_node('rewrite',  rewrite_node)
build.add_node('direct',   direct_answer_node)
build.add_node('generate', generate_node)

build.set_entry_point('classify')
build.add_conditional_edges('classify', route_complexity, {'direct': 'direct', 'retrieve': 'retrieve'})
build.add_edge('retrieve', 'grade')
build.add_conditional_edges('grade', route_grade, {'generate': 'generate', 'rewrite': 'rewrite'})
build.add_edge('rewrite',  'retrieve')
build.add_edge('direct',   END)
build.add_edge('generate', END)

pipeline = build.compile()
print('✅ LangGraph pipeline compiled')

In [ ]:
# ── Run the pipeline ──────────────────────────────────────────────────────────
def run(question: str):
    result = pipeline.invoke({
        'question': question, 'complexity': '', 'documents': [],
        'relevance_grade': '', 'rewritten_query': '', 'iterations': 0,
        'answer': '', 'citations': []
    })
    print(f'\n{"="*65}')
    print(f'Q : {question}')
    print(f'   Complexity : {result["complexity"]}')
    print(f'   Iterations : {result["iterations"]}')
    if result['citations']:
        print(f'   Sources    : {", ".join(set(result["citations"]))}')
    print(f'   Answer     :\n     {result["answer"][:400].strip()}')
    return result

run('What is 2 + 2?')
run('How does RAGAS evaluate a RAG pipeline?')
run('What advanced RAG patterns exist beyond basic retrieval?')

## Step 4 — RAGAS Evaluation

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

# ── Build evaluation samples from pipeline outputs ────────────────────────────
eval_questions = [
    ('What is RAG and why is it used?',
     'RAG retrieves documents and uses them to ground LLM generation, reducing hallucination.'),
    ('How does hybrid search work?',
     'Hybrid search combines BM25 keyword search with dense vector retrieval, merged via RRF.'),
    ('What metrics does RAGAS measure?',
     'RAGAS measures faithfulness, answer relevancy, context precision, and context recall.'),
]

samples = []
for question, reference in eval_questions:
    result   = run(question)
    contexts = [d.page_content for d in result.get('documents', [])]
    if not contexts:
        contexts = ['[No context retrieved — direct answer]']
    samples.append(SingleTurnSample(
        user_input         = question,
        response           = result['answer'],
        retrieved_contexts = contexts,
        reference          = reference,
    ))

dataset = EvaluationDataset(samples=samples)
print(f'\n📊 Built evaluation dataset with {len(samples)} samples')

In [ ]:
# ── Run RAGAS ─────────────────────────────────────────────────────────────────
results = evaluate(
    dataset   = dataset,
    metrics   = [faithfulness, answer_relevancy, context_precision, context_recall],
    llm       = ChatOpenAI(model='gpt-4o-mini'),
    embeddings= embeddings,
)

df = results.to_pandas()
cols = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

print('\n📈 RAGAS Results')
print('='*65)
print(df[['user_input'] + cols].to_string(max_colwidth=40))
print('\n── Mean Scores ──')
for c in cols:
    mean = df[c].mean()
    bar  = '█' * int(mean * 20) + '░' * (20 - int(mean * 20))
    status = '✅' if mean >= 0.7 else '⚠ '
    print(f'  {status} {c:25}: {mean:.4f}  [{bar}]')

## Step 5 — Production Improvements Checklist

### What to Tune if Scores Are Low

| Low Metric | Root Cause | Fix |
|---|---|---|
| **Faithfulness < 0.7** | LLM is hallucinating | Stricter prompt: 'ONLY use context' |
| **Answer Relevancy < 0.7** | Prompt is too vague | Sharpen the answer prompt |
| **Context Precision < 0.7** | Chunks are too noisy | Reduce chunk size or add EmbeddingsFilter |
| **Context Recall < 0.7** | Missing relevant docs | Increase k, use multi-query retriever |

### Performance at Scale
```python
# Async retrieval (for production)
async def aretrieve(query):
    return await retriever.ainvoke(query)

# Semantic cache (avoids re-embedding repeated queries)
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache
set_llm_cache(InMemoryCache())
```

### Monitoring
```python
# LangSmith auto-traces every node
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']    = 'production-rag'
```

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║          🎓 RAG COURSE CAPSTONE COMPLETE                     ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ Module 1  — RAG fundamentals & decision framework        ║
║  ✅ Module 2  — Document loading, splitting & metadata       ║
║  ✅ Module 3  — Embeddings (OpenAI, Ollama, comparison)      ║
║  ✅ Module 4  — Vector stores (Chroma, FAISS, CRUD)          ║
║  ✅ Module 5  — Basic retrieval (similarity, MMR, hybrid)    ║
║  ✅ Module 6  — Advanced retrieval (compression, rerank)     ║
║  ✅ Module 7  — Advanced patterns (fusion, HyDE, CRAG)       ║
║  ✅ Module 8  — Agentic RAG with LangGraph                   ║
║  ✅ Module 9  — RAGAS evaluation suite                       ║
║  ✅ Module 10 — End-to-end production pipeline               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")